# Sparse GP Regression with `gpc`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/sparse_gp_regression.ipynb)

GPc port of [`GPy/sparse_gp_regression.ipynb`](../GPy/sparse_gp_regression.ipynb) (James Hensman / Neil Lawrence).

We show the variational compression effect of sparse GPs using the free-energy framework of [Titsias, 2009](http://jmlr.csail.mit.edu/proceedings/papers/v5/titsias09a/titsias09a.pdf). In GPc the default sparse approximation is **DTCVAR** (`approx="dtcvar"`), matching GPy's VarDTC spirit for this tutorial.


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from gpc.plot import plot_1d

rng = np.random.default_rng(101)

## Sample function

Sample from an RBF prior on a 1-D grid (lengthscale/variance 1), with observation noise.

In [ ]:
N = 50
noise_var = 0.05
X = np.linspace(0, 10, N)[:, None]

def rbf_gram(X, lengthscale=1.0, variance=1.0):
    d2 = (X - X.T) ** 2
    return variance * np.exp(-0.5 * d2 / lengthscale**2)

K = rbf_gram(X)
y = rng.multivariate_normal(np.zeros(N), K + np.eye(N) * noise_var).reshape(-1, 1)

## Full Gaussian process fit

In [ ]:
m_full = gpc.models.GPRegression(X, y, kernel=gpc.kern.RBF(1))
m_full.optimize(iters=200)
plot_1d(m_full, X, y, label="full GP")
plt.title("Full GP")
plt.show()
print(m_full)
print("full objective (−log lik):", m_full.objective())

## A poor sparse GP fit

Inducing inputs clumped in two regions; hold them fixed and set likelihood noise near the truth.

In [ ]:
Z = np.hstack((np.linspace(2.5, 4.0, 3), np.linspace(7.0, 8.5, 3)))[:, None]
m = gpc.models.SparseGPRegression(X, y, Z=Z, kernel=gpc.kern.RBF(1), inducing_fixed=True)
m.set_by_path("likelihood.variance", noise_var)
plot_1d(m, X, y, label="sparse (fixed Z)")
plt.title("Sparse GP, fixed inducing (before optimize)")
plt.show()
print(m)

Fit is reasonable where there are inducing points, but weak elsewhere.

### Optimizing covariance parameters (Z fixed)

Next we optimise the covariance hyperparameters while holding the inducing inputs fixed in their current clumps.


In [ ]:
m.optimize(iters=200)
plot_1d(m, X, y, label="sparse (Z fixed)")
plt.title("After optimizing hypers with Z fixed")
plt.show()
print(m)
print("sparse objective:", m.objective())

The poor location of the inducing inputs causes the model to underfit. The lengthscale is typically much longer than the full GP, and the noise variance larger. In this regime the Kullback–Leibler term in the variational free energy dominates and prefers a longer lengthscale to improve the quality of the approximation — an artefact of badly placed inducing inputs, not of the data.

### Optimizing inducing inputs

Rebuild with free inducing locations (`inducing_fixed=False`, the façade default). We still often see a slightly longer lengthscale than the process we sampled from (or the full GP above).


In [ ]:
m = gpc.models.SparseGPRegression(X, y, Z=Z, kernel=gpc.kern.RBF(1), inducing_fixed=False)
m.optimize(iters=300)
plot_1d(m, X, y, label="sparse (free Z)")
plt.title("After optimizing hypers and Z")
plt.show()
print(m)
print("Z:\n", m.inducing_inputs())
print(
    "lengthscale sparse/full:",
    m.get_by_path("rbf.lengthscale"),
    m_full.get_by_path("rbf.lengthscale"),
)
print(
    "likelihood.variance sparse/full:",
    m.get_by_path("likelihood.variance"),
    m_full.get_by_path("likelihood.variance"),
)


### More inducing points

Twelve inducing inputs should tighten the approximation toward the full GP. When the bound on the marginal likelihood is tight, the variational approximation is good — the gap to the full-GP objective is the KL divergence between the approximation and the truth (up to the usual GPc sign convention on `objective()`).


In [ ]:
Z12 = rng.random((12, 1)) * 12.0
m12 = gpc.models.SparseGPRegression(X, y, Z=Z12, kernel=gpc.kern.RBF(1))
m12.optimize(iters=300)
plot_1d(m12, X, y, label="sparse M=12")
plt.title("Sparse GP, M=12")
plt.show()
plot_1d(m_full, X, y, label="full GP")
plt.title("Full GP (reference)")
plt.show()
print("sparse M=12 objective (−log lik):", m12.objective())
print("full objective (−log lik):       ", m_full.objective())
# GPy compared log_likelihood (maximise); gpc.objective is −log lik (minimise).
print("gap (sparse − full) on −log lik:", m12.objective() - m_full.objective())
print(
    "A small positive gap means a tight variational bound "
    "(approximation close to the full GP)."
)


This time we have enough inducing points and the fit resembles the full GP. That is verified by a tight bound on the marginal likelihood: the difference between the sparse and full objectives is the KL divergence between the variational approximation and the truth (remember `gpc.objective()` is **−log-likelihood**, so compare gaps carefully with GPy's `log_likelihood()`).
